In [ ]:
# Importation
import numpy as np
import pandas as pd
import io
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
# Inspect inputs. inspects supplied input to ascertain they are supposed as expected
# input: supplied input variables
# output: variables for model training. Or fail safe

def inspect_input(model_name, dataset, model_type, path):
    if not isinstance(model_name, str):
        raise TypeError("model name must be a string.")
    if model_type not in ["RF", "SVM"]:
        raise ValueError("model type must be either 'RF' or 'SVM'.")

    try:
        # Attempt to read the CSV file using pandas.
        pd.read_csv(dataset)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"File not found: {dataset}") from e
    except pd.errors.EmptyDataError as e:
        raise ValueError(f"CSV file is empty: {dataset}") from e
    except pd.errors.ParserError as e:
        raise ValueError(f"Invalid CSV file format: {dataset}") from e
    except PermissionError as e:
        raise PermissionError(f"Permission denied: {dataset}") from e

    if not isinstance(path, str):
        raise TypeError("path must be a string representing a system path.")

    return model_name, dataset, model_type, path

In [ ]:
# upload file
uploaded = files.upload()

In [ ]:
#retrieve uploaded filename
file_name = list(uploaded.keys())[0]
print(file_name)

In [ ]:
cvs_file = io.BytesIO(uploaded[file_name])
Mname = "ModelTest"
Mtype = "RF"
Mpath = "/models/"

cvs_file.seek(0)  # Move pointer to the start

model_name, dataset, model_type, path = inspect_input(Mname, cvs_file, Mtype, Mpath)
#print(model_name, model_type, path)
#print(dataset)


In [ ]:
# Process dataset

def process_dataset(dataset):
    # Read the dataset
    dt = pd.read_csv(dataset)

    # Verify the cvs file has the titles requires for model training
    cvs_title = list(dt.columns)
    expected_title = ['subject', 'H.period', 'DD.period.t', 'UD.period.t', 'H.t', 'DD.t.i', 'UD.t.i', 'H.i', 'DD.i.e', 'UD.i.e', 'H.e', 'DD.e.five', 'UD.e.five', 'H.five', 'DD.five.Shift.r', 'UD.five.Shift.r', 'H.Shift.r', 'DD.Shift.r.o', 'UD.Shift.r.o', 'H.o', 'DD.o.a', 'UD.o.a', 'H.a', 'DD.a.n', 'UD.a.n', 'H.n', 'DD.n.l', 'UD.n.l', 'H.l', 'DD.l.Return', 'UD.l.Return', 'H.Return']

    for item in expected_title:
        if item not in cvs_title:
            raise Exception ("The CVS file is not as expected!")

    # Drop subject column
    df = dt[expected_title]
    df = df.drop(columns=['subject'])

    # Calculate row-based mean, variance, and standard deviation
    row_means = df.mean(axis=1)
    row_variances = df.var(axis=1)
    row_std_devs = df.std(axis=1)

    # Add the calculated values as new columns to the DataFrame
    df['row_mean'] = row_means
    df['row_variance'] = row_variances
    df['row_std_dev'] = row_std_devs

    ## Normalize dataframe
    scaler = MinMaxScaler()
    df_normalized = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

    # Add subject column
    df_normalized['subject'] = dt['subject']

    return df_normalized



In [ ]:
dataset.seek(0)
#print(process_dataset(dataset))

In [ ]:
# split training and testing data
# input: normalized dataframe;

def split_train_test(df):
    X = df.drop(columns=['subject'])
    y = df['subject'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% train, 20% test

    return X_train, X_test, y_train, y_test

In [ ]:
# Random forest model

def rf_model(X_train, X_test, y_train, y_test):
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = rf_classifier.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy}")


In [ ]:
# Run code
X_train, X_test, y_train, y_test = split_train_test(process_dataset(dataset))
rf_model(X_train, X_test, y_train, y_test)

In [ ]:
# SVM Model

def svm_model(X_train, X_test, y_train, y_test):
    svm_model = SVC()
    svm_model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = svm_model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy}")

In [ ]:
# Run code
svm_model(X_train, X_test, y_train, y_test)